<a href="https://colab.research.google.com/github/rm-dev-1/adk_dev/blob/main/adk_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import importlib.metadata as md
from google.colab import userdata

print("google-genai:", md.version("google-genai"))  # 入っているバージョンを確認
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

google-genai: 2.12.1


In [3]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")  # 個人のGoogleアカウントで認証する
WORKDIR = Path("/content/drive/MyDrive/gemini-learning")
WORKDIR.mkdir(exist_ok=True)

CACHE_PATH = WORKDIR / "cache.json"
print("保存先:", WORKDIR)

Mounted at /content/drive
保存先: /content/drive/MyDrive/gemini-learning


In [4]:
TICKETS = [
    {"id": 1, "text": "昨夜の日次バッチが失敗していて、今朝のレポートが空です", "cat": "pipeline_failure", "pri": 1},
    {"id": 2, "text": "売上ダッシュボードの数字が先週から更新されていません", "cat": "dashboard_issue", "pri": 2},
    {"id": 3, "text": "マーケ部の新メンバーにBigQueryの閲覧権限をお願いします", "cat": "access_request", "pri": 3},
    {"id": 4, "text": "顧客テーブルに同じ顧客IDが重複して入っているようです", "cat": "data_quality", "pri": 2},
    {"id": 5, "text": "今月のクエリ料金が先月の3倍になっています。原因を知りたい", "cat": "cost_question", "pri": 2},
    {"id": 6, "text": "ストリーミング取り込みが2時間止まっています。至急対応してください", "cat": "pipeline_failure", "pri": 1},
    {"id": 7, "text": "ダッシュボードのフィルタの使い方を教えてください", "cat": "other", "pri": 3},
    {"id": 8, "text": "注文テーブルの金額列にマイナス値が混ざっています", "cat": "data_quality", "pri": 2},
    {"id": 9, "text": "データ分析用のサービスアカウントの権限を拡張したい", "cat": "access_request", "pri": 3},
    {"id": 10, "text": "ダッシュボードが開けず、エラー画面が出ます。役員会議で使います", "cat": "dashboard_issue", "pri": 1},
]
print(len(TICKETS), "件")

10 件


In [5]:
import json
import re
import time
from enum import Enum

import pandas as pd
from pydantic import BaseModel, Field
from google import genai
from google.genai import types, errors

client = genai.Client()

# モデルのカテゴリの解釈を固定化するためのEnum定義
class Category(str, Enum):
    PIPELINE_FAILURE = "pipeline_failure"
    DASHBOARD_ISSUE = "dashboard_issue"
    ACCESS_REQUEST = "access_request"
    DATA_QUALITY = "data_quality"
    COST_QUESTION = "cost_question"
    OTHER = "other"

class Ticket(BaseModel):
    category: Category # 追加情報なし、必須
    priority: int = Field(description="1=緊急(業務停止・期限あり), 2=通常, 3=低") # 追加情報あり、必須
    summary: str = Field(description="30字以内の要約")
    needs_human: bool = Field(description="人間の担当者が対応すべきならtrue")

In [13]:
print(Ticket.__annotations__)

{'category': <enum 'Category'>, 'priority': <class 'int'>, 'summary': <class 'str'>, 'needs_human': <class 'bool'>}


In [10]:
print(Category.PIPELINE_FAILURE)

Category.PIPELINE_FAILURE


In [16]:
MODEL = "gemini-3.5-flash-lite"  # 公式ドキュメントで最新のモデル名を確認

PROMPTS = {
    "baseline": """あなたは社内データ基盤のヘルプデスク担当です。
問い合わせを分類してください。判断に迷う場合は、影響範囲が大きい方を優先します。""",

    "A_definitions": """あなたは社内データ基盤のヘルプデスク担当です。
問い合わせを分類してください。

カテゴリ定義:
- pipeline_failure: バッチ・取り込み・ETLなどの処理の失敗、停止、遅延
- dashboard_issue: ダッシュボードの表示障害、数値の不具合の報告(使い方の質問は含まない)
- access_request: 権限の付与・変更・削除の依頼
- data_quality: データの重複、欠損、不整合
- cost_question: 料金、コスト削減に関する質問
- other: 上記以外(使い方の質問、お礼、一般的な問い合わせなど)""",
}

EXPERIMENT = "baseline"   # ← 実験のたびにここを切り替える
SYSTEM = PROMPTS[EXPERIMENT]

In [17]:
MIN_INTERVAL = 20  # 無料枠5回/分 → 12秒+余裕。制限はモデルごとに違うので公式で確認
CACHE = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
_last_call = 0.0

import random

def classify(text: str, system: str, model: str, retries: int = 5) -> Ticket:
    global _last_call
    key = f"{model}|{system}|{text}"
    if key in CACHE:
        return Ticket.model_validate(CACHE[key])

    for attempt in range(retries):
        wait = MIN_INTERVAL - (time.time() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.time()

        try:
            resp = client.models.generate_content(
                model=model,
                contents=text,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=Ticket,
                    temperature=0,
                ),
            )
            ticket = Ticket.model_validate_json(resp.text)
            CACHE[key] = ticket.model_dump(mode="json")
            CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False))
            return ticket
        except errors.ClientError as e:
            if e.code == 429:
                m = re.search(r"retry in ([\d.]+)s", str(e))
                time.sleep(float(m.group(1)) + 1 if m else 15)
                continue
            raise
        except errors.ServerError as e:
            # 503などサーバー側の混雑:長めに待って再試行
            if attempt == retries - 1:
                raise
            delay = min(10 * 2 ** attempt, 120) + random.uniform(0, 5)
            print(f"\n[{e.code}] {delay:.0f}秒待機(試行{attempt + 1}/{retries})")
            time.sleep(delay)
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("リトライ上限に達しました")

In [18]:
rows = []
for i, t in enumerate(TICKETS, 1):
    print(f"{i}/{len(TICKETS)}", end=" ", flush=True)
    r = classify(t["text"], SYSTEM, MODEL)
    rows.append({
        "id": t["id"], "text": t["text"],
        "expected_cat": t["cat"], "expected_pri": t["pri"],
        "pred_cat": r.category.value, "pred_pri": r.priority,
        "summary": r.summary, "needs_human": r.needs_human,
    })

df = pd.DataFrame(rows)
df.to_csv(WORKDIR / f"result_{EXPERIMENT}.csv", index=False)
print("\n完了")

1/10 2/10 3/10 4/10 5/10 6/10 7/10 8/10 9/10 10/10 
完了


In [19]:
n = len(df)
cat_ok = (df.expected_cat == df.pred_cat).sum()
pri_ok = (df.expected_pri == df.pred_pri).sum()
print(f"[{EXPERIMENT}] カテゴリ正解: {cat_ok}/{n} ({cat_ok/n:.3f})")
print(f"[{EXPERIMENT}] 優先度正解: {pri_ok}/{n} ({pri_ok/n:.3f})")
print("優先度MAE:", round((df.expected_pri - df.pred_pri).abs().mean(), 3))

urgent = df[df.expected_pri == 1]
if len(urgent):
    miss = (urgent.pred_pri != 1).sum()
    print(f"緊急の見逃し: {miss}/{len(urgent)}")

print("\n--- カテゴリ混同行列 ---")
print(pd.crosstab(df.expected_cat, df.pred_cat))

print("\n--- カテゴリの誤り ---")
print(df[df.expected_cat != df.pred_cat][["id", "text", "expected_cat", "pred_cat"]].to_string())

print("\n--- 優先度の誤り ---")
print(df[df.expected_pri != df.pred_pri][["id", "text", "expected_pri", "pred_pri"]].to_string())

print("\n--- 優先度混同行列 ---")
print(pd.crosstab(df.expected_pri, df.pred_pri))

df["summary_len"] = df.summary.str.len()
print("\n--- summaryの文字数 ---")
print(df.summary_len.describe())
print("30字超:", (df.summary_len > 30).sum(), "件")

[baseline] カテゴリ正解: 9/10 (0.900)
[baseline] 優先度正解: 6/10 (0.600)
優先度MAE: 0.4
緊急の見逃し: 0/3

--- カテゴリ混同行列 ---
pred_cat          access_request  cost_question  dashboard_issue  \
expected_cat                                                       
access_request                 2              0                0   
cost_question                  0              1                0   
dashboard_issue                0              0                2   
data_quality                   0              0                0   
other                          0              0                1   
pipeline_failure               0              0                0   

pred_cat          data_quality  pipeline_failure  
expected_cat                                      
access_request               0                 0  
cost_question                0                 0  
dashboard_issue              0                 0  
data_quality                 2                 0  
other                        0              